# RE2.1 · Cifras del diseño: la arquitectura propuesta y las líneas base

Lo que el documento de diseño toma del código, para que no se desincronice: hiperparámetros del
AST-Deformable-DETR, tamaño de cada bloque, resolución de la pirámide frente a la duración de las
clases, y las recetas de entrenamiento de las cuatro configuraciones (`train.py: PRESETS`).

In [1]:
import json
import re
from dataclasses import asdict

import pandas as pd
import reporte
import torch
from reporte import PROJECT_DIR, number

from core.config import MAX_DETECTIONS, NMS_IOU, SCORE_FLOOR, P
from data.annotations import load_annotations
from evaluation.metrics import MATCH_IOU, WINDOW_CLASS_WIDTH
from evaluation.protocol import MIN_PRECISIONS, N_BOOTSTRAP
from models import criterion, deformable_detr
from models.backbone import AST_CHECKPOINT, N_LEVELS
from models.coco_deformable_detr import DETR_CHECKPOINT
from models.deformable_detr import ASTDeformableDETR
from models.faster_rcnn import ANCHOR_RATIOS, ANCHOR_SIZES, TRAINABLE_LAYERS
from models.registry import build_model
from models.yolo import IMAGE_SIZE
from prepare_data import select_experiment
from train import PRESETS
from training.yolo import AUGMENTATION, PATIENCE

OUT = reporte.out_dir("RE_2-1")
N_CLASSES = 25
# Sexta configuración: la propuesta con la cabeza preentrenada sobre cantos de aves
# (`notebooks/pretrain_birds_train.py`), con su caché y su corrida aparte.
BIRDS_CACHE = PROJECT_DIR / "data" / "processed_extra"
BIRDS_RUN = PROJECT_DIR / "runs_extra" / "detr_birds"
EPOCH_LINE = re.compile(r"\[(?P<epoch>\d+)/(?P<total>\d+)\].*mAP30=(?P<map>[0-9.]+)")
NAMES = {
    "detr": "AST-Deformable-DETR (propuesta)",
    "detr_resnet": "ResNet-50 Deformable DETR (COCO)",
    "frcnn": "Faster R-CNN R50-FPN v2 (COCO)",
    "yolo": "YOLO26s (COCO)",
}

In [2]:
# La propuesta, tal como la arma `train.py --arch detr` con el frontend logmel.
hparams = PRESETS["detr"].hparams | {"frontend": "logmel"}
model = build_model(PRESETS["detr"].arch, N_CLASSES, hparams)
assert isinstance(model, ASTDeformableDETR)
backbone, head = model.backbone, model.head


def millions(*prefixes: str) -> float:
    # Parámetros (en millones) de los submódulos cuyo nombre empieza por alguno de los prefijos.
    return (
        sum(p.numel() for n, p in model.named_parameters() if n.startswith(prefixes or ("",))) / 1e6
    )


blocks = {
    "Front-end (log-mel, sin parámetros)": ("frontend.",),
    "AST (12 capas, 768-d)": ("backbone.",),
    "Proyección 768 $\\to$ 128": ("head.proj.",),
    "Pirámide multiescala": ("head.pyramid.",),
    "Decodificador deformable (6 capas)": ("head.detr.layers.",),
    "Consultas y punto de referencia": (
        "head.detr.query_embed.",
        "head.detr.query_pos.",
        "head.detr.ref_point_head.",
    ),
    "Cabezas de caja (6)": ("head.detr.bbox_heads.",),
    "Cabezas de clase (6 $\\times$ 25)": ("head.detr.class_heads.",),
}
sizes = pd.Series({name: millions(*prefixes) for name, prefixes in blocks.items()})
total = millions()
reporte.write_rows(
    OUT / "bloques.tex",
    [
        (name, number(size, 2), number(100 * size / total, 1) + "\\,\\%")
        for name, size in sizes.items()
    ]
    + [("\\textbf{Total}", number(total, 1), "100\\,\\%")],
)
print(f"{total:.1f} M parámetros · tokens {backbone.freq_out} × {backbone.time_out}")
sizes.round(3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

88.6 M parámetros · tokens 12 × 32


Front-end (log-mel, sin parámetros)     0.000
AST (12 capas, 768-d)                  85.551
Proyección 768 $\to$ 128                0.098
Pirámide multiescala                    0.263
Decodificador deformable (6 capas)      2.476
Consultas y punto de referencia         0.026
Cabezas de caja (6)                     0.201
Cabezas de clase (6 $\times$ 25)        0.019
dtype: float64

In [3]:
# Resolución de cada nivel de la pirámide en tiempo (ms por celda) y en bandas mel por celda.
scales = [4, 2, 1, 0.5]
levels = pd.DataFrame(
    {
        "escala": [f"{number(s, 1) if s < 1 else number(s)}$\\times$" for s in scales],
        "alto": [int(backbone.freq_out * s) for s in scales],
        "ancho": [int(backbone.time_out * s) for s in scales],
    }
)
levels["ms por celda"] = P.clip_len_s * 1000 / levels["ancho"]
levels["bandas mel por celda"] = P.n_mels / levels["alto"]
reporte.write_rows(
    OUT / "niveles.tex",
    [
        (
            r.escala,
            f"{r.alto} $\\times$ {r.ancho}",
            number(r["ms por celda"]),
            number(r["bandas mel por celda"], 1),
        )
        for _, r in levels.iterrows()
    ],
)
levels

,escala,alto,ancho,ms por celda,bandas mel por celda
0,4$\times$,48,128,23.4375,2.666667
1,2$\times$,24,64,46.8750,5.333333
2,1$\times$,12,32,93.7500,10.666667
3,"0,5$\times$",6,16,187.5000,21.333333


In [4]:
# Duración de cada clase (cuartiles) frente a la resolución temporal de los niveles.
annotations = load_annotations()
experiment, labels = select_experiment(annotations)
durations = experiment.groupby("label")["duration_s"]
quartiles = (
    pd.DataFrame(
        {"q1": durations.quantile(0.25), "med": durations.median(), "q3": durations.quantile(0.75)}
    )
    * 1000
)
quartiles = quartiles.sort_values("med").rename_axis("clase").reset_index()
reporte.write_data(OUT / "duraciones", quartiles.round(1))
reporte.write_macros(
    OUT / "resolucion.tex",
    {
        f"cell_{name}": number(ms)
        for name, ms in zip(("zero", "one", "two", "three"), levels["ms por celda"], strict=True)
    },
)
quartiles.round(0).T

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
clase,sm/cc,aa/gc,sb/spc,sm/fs,sb/ppc,aa/hm,aa/sc,sm/hic,pt/sqc,as/bc,...,lw/trino,sm/pc,sm/sc,sb/pcc,lw/sqc,ac/gc,ac/sc,ac/chc,pt/dc,as/hc
q1,64.0,77.0,127.0,148.0,146.0,195.0,212.0,186.0,237.0,259.0,...,354.0,404.0,355.0,496.0,605.0,657.0,867.0,865.0,4759.0,7129.0
med,81.0,93.0,162.0,189.0,205.0,228.0,232.0,266.0,267.0,322.0,...,534.0,550.0,586.0,726.0,812.0,1012.0,1180.0,1189.0,6678.0,11024.0
q3,107.0,118.0,221.0,260.0,303.0,267.0,266.0,364.0,314.0,397.0,...,730.0,823.0,822.0,1059.0,1048.0,1922.0,1422.0,1583.0,9549.0,15871.0


In [5]:
# Recetas de entrenamiento de las cuatro configuraciones, desde `train.py: PRESETS`.
def scientific(value: float) -> str:
    mantissa, exponent = f"{value:.0e}".split("e")
    return f"${mantissa} \\times 10^{{{int(exponent)}}}$"


rows = []
for key, preset in PRESETS.items():
    config = asdict(preset.config)
    rows.append(
        (
            NAMES[key],
            number(config["epochs"]),
            number(config["batch_size"]),
            scientific(config["learning_rate"]) if key != "yolo" else "auto",
            "AdamW + OneCycle" if key != "yolo" else "Ultralytics (cos)",
            "mAP@0,3 en val" if key != "yolo" else "aptitud de Ultralytics",
        )
    )
reporte.write_rows(OUT / "recetas.tex", rows)
pd.DataFrame(rows, columns=["configuración", "épocas", "lote", "lr", "optimizador", "checkpoint"])

,configuración,épocas,lote,lr,optimizador,checkpoint
0,AST-Deformable-DETR (propuesta),30,8,$2 \times 10^{-4}$,AdamW + OneCycle,"mAP@0,3 en val"
1,ResNet-50 Deformable DETR (COCO),30,8,$2 \times 10^{-4}$,AdamW + OneCycle,"mAP@0,3 en val"
2,Faster R-CNN R50-FPN v2 (COCO),12,4,$1 \times 10^{-4}$,AdamW + OneCycle,"mAP@0,3 en val"
3,YOLO26s (COCO),30,32,auto,Ultralytics (cos),aptitud de Ultralytics


In [6]:
# El preentrenamiento con aves: clases y ventanas del caché aparte y mejor época de la corrida.
bird_labels = json.loads((BIRDS_CACHE / "labels.json").read_text())
bird_meta = json.loads((BIRDS_CACHE / "meta.json").read_text())
bird_windows = {
    split: len(torch.load(BIRDS_CACHE / f"{split}.pt", mmap=True, weights_only=True)["labels"])
    for split in ("train", "val")
}
bird_epochs = [
    m
    for line in (BIRDS_RUN / "train.log").read_text().splitlines()
    if (m := EPOCH_LINE.search(line))
]
bird_best = max(bird_epochs, key=lambda m: float(m["map"]))
bird_config = json.loads((BIRDS_RUN / "config.json").read_text())
birds = {
    "n_bird_classes": number(len(bird_labels)),
    "n_bird_species": number(sum(1 for name in bird_labels.values() if name.startswith("pow/"))),
    "n_bird_windows_train": number(bird_windows["train"]),
    "n_bird_windows_val": number(bird_windows["val"]),
    "bird_sources": ", ".join(sorted(bird_meta["sources"])),
    "bird_epochs": number(int(bird_epochs[-1]["total"])),
    "bird_best_epoch": number(int(bird_best["epoch"])),
    "bird_best_map": number(float(bird_best["map"]), 3),
    "bird_split_train": number(100 * bird_meta["split_ratios"][0], 0),
    "bird_split_val": number(100 * bird_meta["split_ratios"][1], 0),
}
assert bird_config["hparams"] == hparams, "el preentrenamiento usa otros hiperparámetros"
pd.Series(birds)

n_bird_classes                            22
n_bird_species                            21
n_bird_windows_train                 43\,707
n_bird_windows_val                   11\,276
bird_sources            powdermill, pteroset
bird_epochs                               30
bird_best_epoch                           24
bird_best_map                          0,831
bird_split_train                          85
bird_split_val                            15
dtype: str

In [7]:
reporte.write_macros(
    OUT / "valores.tex",
    {
        "n_params": number(total, 1),
        "n_params_backbone": number(millions("backbone."), 1),
        "n_params_head": number(millions("head."), 2),
        "tokens_freq": number(backbone.freq_out),
        "tokens_time": number(backbone.time_out),
        "hidden": number(backbone.hidden_size),
        "time_stride": number(int(hparams["time_stride"])),
        "n_frames": number(P.n_frames),
        "n_mels": number(P.n_mels),
        "head_dim": number(deformable_detr.DIM),
        "n_queries": number(deformable_detr.N_QUERIES),
        "n_decoder_layers": number(deformable_detr.N_DECODER_LAYERS),
        "n_heads": number(deformable_detr.N_HEADS),
        "n_points": number(deformable_detr.N_POINTS),
        "n_levels": number(N_LEVELS),
        "ffn": number(deformable_detr.FFN),
        "dropout": number(deformable_detr.DROPOUT, 1),
        "prior_prob": number(deformable_detr.PRIOR_PROB, 2),
        "focal_alpha": number(criterion.FOCAL_ALPHA, 2),
        "focal_gamma": number(criterion.FOCAL_GAMMA, 1),
        "cost_class": number(criterion.COST_CLASS),
        "cost_bbox": number(criterion.COST_BBOX),
        "cost_iou": number(criterion.COST_IOU),
        "weight_class": number(criterion.WEIGHT_CLASS),
        "ast_checkpoint": AST_CHECKPOINT.replace("_", "\\_").replace("-", "-\\allowbreak{}"),
        "detr_checkpoint": DETR_CHECKPOINT.replace("_", "\\_").replace("-", "-\\allowbreak{}"),
        "anchor_ratios": ", ".join(number(r, 2) for r in ANCHOR_RATIOS),
        "anchor_sizes": ", ".join(number(s[0]) for s in ANCHOR_SIZES),
        "trainable_layers": number(TRAINABLE_LAYERS),
        "yolo_image_size": number(IMAGE_SIZE),
        "yolo_patience": number(PATIENCE),
        "yolo_scale": number(AUGMENTATION["scale"], 1),
        "yolo_translate": number(AUGMENTATION["translate"], 2),
        "nms_iou": number(NMS_IOU, 1),
        "max_detections": number(MAX_DETECTIONS),
        "match_iou": number(MATCH_IOU, 1),
        "score_floor": number(SCORE_FLOOR, 3),
        "window_class_width": number(WINDOW_CLASS_WIDTH, 2),
        "min_precision": number(MIN_PRECISIONS[0], 2),
        "min_precision_alt": number(MIN_PRECISIONS[1], 2),
        "n_bootstrap": number(N_BOOTSTRAP),
        **birds,
        "epochs": number(PRESETS["detr"].config.epochs),
        "batch_size": number(PRESETS["detr"].config.batch_size),
        "learning_rate": scientific(PRESETS["detr"].config.learning_rate),
        "weight_decay": scientific(PRESETS["detr"].config.weight_decay),
        "warmup": number(100 * PRESETS["detr"].config.warmup) + "\\,\\%",
        "clip_grad": number(ASTDeformableDETR.clip_grad, 1),
    },
)
sorted(OUT.iterdir())

[PosixPath('/home/fcandia/tesis-primate/research/reportes/figures/RE_2-1/bloques.tex'),
 PosixPath('/home/fcandia/tesis-primate/research/reportes/figures/RE_2-1/duraciones.dat'),
 PosixPath('/home/fcandia/tesis-primate/research/reportes/figures/RE_2-1/niveles.tex'),
 PosixPath('/home/fcandia/tesis-primate/research/reportes/figures/RE_2-1/recetas.tex'),
 PosixPath('/home/fcandia/tesis-primate/research/reportes/figures/RE_2-1/resolucion.tex'),
 PosixPath('/home/fcandia/tesis-primate/research/reportes/figures/RE_2-1/valores.tex')]